# Simulating a Field of Stars — Liger IFS

In [42]:
from liger_iris_sim.sources import make_point_source_ifs_cube
from liger_iris_sim.expose import compute_liger_throughput, expose_ifs
from liger_iris_sim.sky import get_maunakea_spectral_sky_emission, get_maunakea_spectral_sky_transmission
from liger_iris_sim.utils import LIGER_PROPS, rebin_image, compute_filter_photon_flux, generate_wave_grid_for_filter

from liger_iris_drp_resources.filters import load_filters_summary
from liger_iris_drp_resources.psfs import get_liger_psf, download_liger_psfs
from liger_iris_drp_resources.model_spectra import download_model_spectra

import numpy as np
import matplotlib.pyplot as plt

## Download required resources

PSF and model spectra files are downloaded automatically to the local resources directory on first use.

In [43]:
download_liger_psfs()
download_model_spectra()

'/Users/cale/.astropy/cache/download/url/LIGER_IRIS_DRP_RESOURCES/Model_Spectra'

## Instrument and exposure parameters

In [44]:
np.random.seed(1)

mode = 'ifs'
ifs_mode = 'lenslet'  # 'lenslet' or 'slicer'
filt = 'J'
size = (128, 128)       # IFS spatial dimensions (y, x) in spaxels
read_noise = 9        # e- RMS
dark_current = 0.025  # e- / sec / pixel
scale = 0.014         # arcsec / spaxel (lenslet)
itime = 30           # integration time, sec
n_frames = 1          # number of coadded frames
resolution = 4000     # spectral resolution R = lambda / delta_lambda
collarea = LIGER_PROPS['keck_collarea']  # m^2

## Load filter data and compute throughput

`load_filters_summary` returns filter metadata including the central wavelength, zero-point flux, and background magnitude.
`compute_liger_throughput` folds together telescope, AO, filter, and instrument throughput.

In [45]:
filter_info = load_filters_summary(filter_name=filt)

tput = compute_liger_throughput(
    mode=mode,
    wave=filter_info['wavecenter'],
    ifs_mode=ifs_mode,
)
print(f"Total throughput for {ifs_mode} mode at {filter_info['wavecenter']:.3f} µm: {tput:.3f}")

Total throughput for lenslet mode at 1.248 µm: 0.214


## Build the wavelength grid

`generate_wave_grid_for_filter` returns a Nyquist-sampled wavelength array covering the filter bandpass at the requested spectral resolution.

In [46]:
wave = generate_wave_grid_for_filter(filter_info, resolution=resolution)
print(f"Wavelength grid: {wave[0]:.4f} – {wave[-1]:.4f} µm, {len(wave)} channels")

Wavelength grid: 1.1660 – 1.3301 µm, 1127 channels


## Load and rebin the on-axis PSF

The PSF is loaded at the filter central wavelength and rebinned to the IFS spaxel scale.

In [47]:
psf, psf_info = get_liger_psf(filter_info['wavecenter'], xs=0, ys=0)
psf = rebin_image(
    psf,
    scale_in=psf_info['psf_sampling'],
    scale_out=scale,
)
print(f"PSF shape after rebinning: {psf.shape}")

PSF shape after rebinning: (80, 80)


## Compute sky emission and transmission

`get_maunakea_spectral_sky_emission` returns the per-spaxel sky emission spectrum in photons / sec / m² / wavebin.
`get_maunakea_spectral_sky_transmission` returns the normalised atmospheric transmission spectrum (0–1).
Both are sampled on the same wavelength grid as the source cube.

In [48]:
sky_em = get_maunakea_spectral_sky_emission(
    wave, resolution=resolution,
    T_tel=275, T_atm=258, T_aos=243, T_zod=5800,
    Em_tel=0.09, Em_atm=0.2, Em_aos=0.01,
)
# Integrate over spaxel solid angle: photons / (s * m^2 * wavebin)
sky_em_rate = sky_em['sky_em'] * scale**2

sky_trans = get_maunakea_spectral_sky_transmission(wave, resolution=resolution, airmass=1)

## Create a field of stars

Each star has a flat spectrum normalised so that its total photon flux matches the requested magnitude.
`make_point_source_ifs_cube` places each star into the source cube, convolved spaxel-by-spaxel with the PSF.

In [49]:
n_stars = 10
mag_range = (16, 17)

input_cube_rate = np.zeros((len(wave), *size), dtype=np.float32)

for i in range(n_stars):
    mag = np.random.uniform(mag_range[0], mag_range[1])
    xpix = int(np.random.uniform(5, size[1] - 5))
    ypix = int(np.random.uniform(5, size[0] - 5))
    photon_flux = compute_filter_photon_flux(mag, zp=filter_info['zpphot'])  # photons / sec / m^2

    # Flat spectrum normalised to total photon flux
    flat_spec = np.ones(len(wave), dtype=np.float32)
    template_flux = flat_spec / flat_spec.sum() * photon_flux
    template = (wave, template_flux)

    print(f"Star {i+1:2d}: mag={mag:.2f}, x={xpix}, y={ypix}, flux={photon_flux:.2e} phot/s/m²")
    make_point_source_ifs_cube(
        xpix, ypix, template,
        psf=psf,
        cube_out=input_cube_rate,
    )

Star  1: mag=16.42, x=89, y=5, flux=1.24e+03 phot/s/m²
Creating IFS cube with point source xdet=89, ydet=5, -0.00014574999999994454 microns, num wavelengths 1127
Star  2: mag=16.30, x=22, y=15, flux=1.38e+03 phot/s/m²
Creating IFS cube with point source xdet=22, ydet=15, -0.00014574999999994454 microns, num wavelengths 1127
Star  3: mag=16.19, x=45, y=51, flux=1.53e+03 phot/s/m²
Creating IFS cube with point source xdet=45, ydet=51, -0.00014574999999994454 microns, num wavelengths 1127
Star  4: mag=16.54, x=54, y=85, flux=1.11e+03 phot/s/m²
Creating IFS cube with point source xdet=54, ydet=85, -0.00014574999999994454 microns, num wavelengths 1127
Star  5: mag=16.20, x=108, y=8, flux=1.51e+03 phot/s/m²
Creating IFS cube with point source xdet=108, ydet=8, -0.00014574999999994454 microns, num wavelengths 1127
Star  6: mag=16.67, x=54, y=70, flux=9.80e+02 phot/s/m²
Creating IFS cube with point source xdet=54, ydet=70, -0.00014574999999994454 microns, num wavelengths 1127
Star  7: mag=16.14

## Run the IFS exposure simulation

`expose_ifs` applies throughput, sky emission, sky transmission, Poisson noise, dark current, and read noise to produce a simulated data cube.

In [50]:
sim = expose_ifs(
    input_cube_rate,
    itime=itime, n_frames=n_frames, collarea=collarea,
    sky_emission_rate=sky_em_rate,
    sky_transmission=sky_trans['sky_trans'],
    tput=tput, read_noise=read_noise, dark_current=dark_current,
)

print(f"Output cube shape: {sim['observed_tot'].shape}")
print(f"Peak SNR (per voxel): {np.nanmax(sim['snr']):.1f}")

Output cube shape: (1127, 128, 128)
Peak SNR (per voxel): 15.2


## Visualize results

Collapse the cube along the wavelength axis to produce a white-light image, and show the peak-SNR spectrum.

In [ ]:
snr_map = np.nansum(sim['snr']**2, axis=0)**0.5
im = plt.imshow(snr_map, origin='lower', cmap='viridis', vmin=0)
plt.colorbar(im, label='SNR')
plt.title('SNR integrated over bandpass')
plt.show()

In [ ]:
# Extract and plot the spectrum of the brightest spaxel
peak_idx = np.unravel_index(np.nanargmax(snr_map), snr_map.shape)
ypeak, xpeak = peak_idx

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(wave, sim['observed_rate'][:, ypeak, xpeak], lw=1, label='Observed')
axes[0].plot(wave, sim['source_rate'][:, ypeak, xpeak], lw=1, ls='--', label='Source (noiseless)')
axes[0].set_ylabel('Signal (e-/sec)')
axes[0].legend()
axes[0].set_title(f'Spectrum at brightest spaxel (x={xpeak}, y={ypeak})')

axes[1].plot(wave, sim['snr'][:, ypeak, xpeak], color='C2', lw=1)
axes[1].set_ylabel('SNR')
axes[1].set_xlabel('Wavelength (µm)')
plt.show()

Text(0.5, 0, 'Wavelength (µm)')